# Profiling the sources

What the public data can and cannot say about the rule's two milestones.
Written up in [`../research/profile_findings.md`](../research/profile_findings.md).


In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

import fasttrack as ft

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 80)


## Source freshness

The cycle closed 2026-06-30. Which sources actually reach it?


In [2]:
import collect

collect.report_freshness()



Cycle under measurement: 2021-07-01 .. 2026-06-30
  hg8x-zxpr  2026-05-19  Affordable Housing Production by Building <-- ends before cycle end


  dbdt-5s7j  2026-03-16  Housing Database by Community District <-- ends before cycle end
  48dt-mn3z  2026-03-16  Housing Database by 2020 CDTA <-- ends before cycle end


  br6q-ssj3  2026-03-16  Housing Database Project Level Files <-- ends before cycle end
  rbx6-tga4  2026-09-18  DOB NOW: Build – Approved Permits


  ipu4-2q9a  2026-09-18  DOB Permit Issuance


## The permit join

The rule makes a DOB construction permit a *condition*, so an unmatched
building drops out of the numerator - which lowers a district's rate and makes
it more likely to be listed. That is a bias toward the finding this project is
looking for, so it gets checked first.


In [3]:
hpd = ft.attach_permits(ft.load_hpd(), *ft.load_permit_dates())

in_cycle = hpd[
    (hpd["project_start_date"] >= ft.CYCLE_START)
    & (hpd["construction_type"] == ft.NEW_CONSTRUCTION)
]
in_cycle["permit_source"].value_counts()


permit_source
bbl          1853
unmatched     561
bin             9
Name: count, dtype: int64

### Named projects match; redacted ones cannot

HPD writes `CONFIDENTIAL` in place of the project name and blanks the BBL,
BIN and address for some rows - almost all of them single-unit. Those rows are
unmatchable by construction, not because the join is weak.


In [4]:
confidential = in_cycle["project_name"].fillna("").str.upper().eq("CONFIDENTIAL")

pd.DataFrame(
    {
        "rows": [(~confidential).sum(), confidential.sum()],
        "units": [
            in_cycle.loc[~confidential, ft.UNIT_COLUMN].sum(),
            in_cycle.loc[confidential, ft.UNIT_COLUMN].sum(),
        ],
        "pct_unmatched": [
            100 * (in_cycle.loc[~confidential, "permit_source"] == "unmatched").mean(),
            100 * (in_cycle.loc[confidential, "permit_source"] == "unmatched").mean(),
        ],
    },
    index=["named", "confidential"],
).round(1)


,rows,units,pct_unmatched
named,1925,62573.0,3.3
confidential,498,572.0,100.0


Where the redacted rows sit matters more than how many there are - the whole
question is the bottom of a ranking.


In [5]:
in_cycle[confidential].groupby("district")[ft.UNIT_COLUMN].agg(["size", "sum"]).sort_values(
    "sum", ascending=False
).head(12)


,size,sum
district,,
MN-03,5,79.0
SI-01,52,52.0
BK-18,24,24.0
BX-09,23,23.0
SI-03,22,22.0
BX-12,17,17.0
MN-09,16,16.0
BX-07,16,16.0
BX-11,16,16.0


### Projects the permit timing excludes

The permit must be issued by cycle end. These cleared in July.


In [6]:
late = in_cycle[
    (~confidential)
    & in_cycle["permit_date"].notna()
    & (in_cycle["permit_date"] > ft.CYCLE_END)
]
late[["district", "project_name", "project_start_date", "permit_date", ft.UNIT_COLUMN]]


,district,project_name,project_start_date,permit_date,all_counted_units
128,MN-07,WSFSSH. 105 WEST 108 STREET,2026-01-30,2026-07-15,84.0
185,BK-05,SUTTER PLACE,2025-12-24,2026-07-13,6.0


## The denominator

`dbdt-5s7j` reports identical 2020 stock for two pairs of districts. Checked
against the same pipeline run on CDTA geography.


In [7]:
ft.denominator_discrepancies()


,cd_census_units_2020,cd_net_new_units,cd_housing_units,cdta_census_units_2020,cdta_net_new_units,cdta_housing_units,pct_difference,duplicated_in_cd_file
district,,,,,,,,
MN-05,42323,509,42832,39593,544,40137,-0.064504,False
BK-15,67322,468,67790,64571,468,65039,-0.040863,False
BK-09,41694,1090,42784,40871,1090,41961,-0.019739,False
QN-08,59385,520,59905,58218,553,58771,-0.019651,False
BK-02,66998,4232,71230,65710,3300,69010,-0.019224,False
MN-04,84357,759,85116,82889,759,83648,-0.017402,False
BX-12,60856,585,61441,59856,584,60440,-0.016432,False
BX-03,34360,1033,35393,33823,1034,34857,-0.015629,False
BK-10,55768,56,55824,55045,49,55094,-0.012964,False
